In [ ]:
import pandas as pd
import numpy as np
from nflfastpy import load_pbp   # ← most common Python port / wrapper
# Alternative: from nfl_data_py import import_pbp_data (very similar API)

In [ ]:
# Cell 2 - Load play-by-play data (2018–2025)
# Note: seasons are passed as a list or range
pbp_all = load_pbp(list(range(2018, 2026)))   # 2025 season included if available
# Alternative with nfl_data_py:
# pbp_all = import_pbp_data(years=range(2018, 2026))

print(f"Loaded {len(pbp_all):,} plays from {pbp_all.season.min()}–{pbp_all.season.max()}")
pbp_all.head(3)

In [ ]:
# Cell 3 - Create penalties and fourth downs dataframes
penalties = (
    pbp_all
    .query("penalty == 1")
    .filter(like="penalty", axis="columns")
    .join(pbp_all[["ep", "epa", "desc", "play_type", "play_clock"]])
    .join(pbp_all.drop(columns=pbp_all.filter(like="penalty").columns.union(["ep", "epa", "desc", "play_type", "play_clock"])))
)

fourth_downs = pbp_all.query("down == 4")

print("Penalties shape:", penalties.shape)
print("Fourth downs shape:", fourth_downs.shape)

In [ ]:
# Cell 4 - Defensive jumps / offside penalties
jumps = penalties[
    penalties["penalty_type"].isin(["Defensive Offside", "Encroachment", "Neutral Zone Infraction"])
]

# Jumps on 4th down & short yardage (≤ 5 yards)
jumps_4d = jumps.query("down == 4 & ydstogo <= 5")

# Same, but trying to exclude punts / field goals (most conservative filter)
jumps_4d_off = jumps_4d[
    ~jumps_4d["desc"].str.contains(r"(?i)(Punt|Field Goal)", regex=True)
]

print("All defensive jumps:          ", len(jumps))
print("Jumps on 4th & ≤5:            ", len(jumps_4d))
print("Jumps on 4th & ≤5 (no punt/FG):", len(jumps_4d_off))

In [ ]:
# Cell 5 - Delay of Game on 4th & short (non-special teams)
jumps_fail_delays = penalties[
    (penalties["penalty_type"] == "Delay of Game") &
    (penalties["down"] == 4) &
    (penalties["ydstogo"] <= 5) &
    (~penalties["desc"].str.contains(r"(?i)(Punt|Field Goal)", regex=True))
]

# Cell 6 - Attempts to find timeout → 4th & short situations
# Version 1: timeout this play and next play is 4th down
jumps_fail_to = (
    pbp_all
    .assign(
        next_down=pbp_all["down"].shift(-1),
        next_ydstogo=pbp_all["ydstogo"].shift(-1)
    )
    .query("timeout == 1 & next_down == 4 & next_ydstogo <= 5")
)

print("Timeouts immediately before 4th & ≤5:", len(jumps_fail_to))